In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
os.chdir('/content/drive/MyDrive/credit-risk-assessment-system')

import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, classification_report

X_train = pd.read_csv('data/processed/X_train_fe.csv')
X_test = pd.read_csv('data/processed/X_test_fe.csv')
y_train = pd.read_csv('data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('data/processed/y_test.csv').squeeze()

leftover_cols = [c for c in ['SK_ID_CURR', 'AMT_ANNUITY_RAW', 'AMT_CREDIT_RAW', 'AMT_INCOME_TOTAL_RAW']
                  if c in X_train.columns]
X_train_model = X_train.drop(columns=leftover_cols)
X_test_model = X_test.drop(columns=leftover_cols)

print(X_train_model.shape, X_test_model.shape)

(246008, 191) (61503, 191)


In [3]:
!pip install catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.2 MB/s eta 0:00:000:00:0100:01


In [4]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='auc'
)
xgb.fit(X_train_model, y_train)

y_pred_proba_xgb = xgb.predict_proba(X_test_model)[:, 1]
auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)
print(f"XGBoost AUC-ROC: {auc_xgb:.4f}")

XGBoost AUC-ROC: 0.7615


In [6]:
import re

def clean_column_names(df):
    df = df.copy()
    df.columns = [re.sub(r'[^A-Za-z0-9_]+', '_', col) for col in df.columns]
    return df

X_train_lgbm = clean_column_names(X_train_model)
X_test_lgbm = clean_column_names(X_test_model)

print(X_train_lgbm.shape, X_test_lgbm.shape)

(246008, 191) (61503, 191)


In [8]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)
lgbm.fit(X_train_lgbm, y_train)

y_pred_proba_lgbm = lgbm.predict_proba(X_test_lgbm)[:, 1]
auc_lgbm = roc_auc_score(y_test, y_pred_proba_lgbm)
print(f"LightGBM AUC-ROC: {auc_lgbm:.4f}")

[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.210636 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5464
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 181
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432482
[LightGBM] [Info] Start training from score -2.432482
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
LightGBM AUC-ROC: 0.7606


In [9]:
print(f"LightGBM AUC-ROC: {auc_lgbm:.4f}")

LightGBM AUC-ROC: 0.7606


In [10]:
from catboost import CatBoostClassifier

catboost = CatBoostClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    verbose=0
)
catboost.fit(X_train_model, y_train)

y_pred_proba_cb = catboost.predict_proba(X_test_model)[:, 1]
auc_cb = roc_auc_score(y_test, y_pred_proba_cb)
print(f"CatBoost AUC-ROC: {auc_cb:.4f}")

CatBoost AUC-ROC: 0.7611


In [11]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'XGBoost', 'LightGBM', 'CatBoost'],
    'AUC-ROC': [0.7410, 0.7186, auc_xgb, auc_lgbm, auc_cb]
})
results = results.sort_values('AUC-ROC', ascending=False).reset_index(drop=True)
print(results)

results.to_csv('reports/model_comparison_results.csv', index=False)
print("Saved successfully")

                 Model   AUC-ROC
0              XGBoost  0.761483
1             CatBoost  0.761144
2             LightGBM  0.760599
3  Logistic Regression  0.741000
4        Decision Tree  0.718600
Saved successfully


In [13]:
os.makedirs('models', exist_ok=True)

In [14]:
import joblib

joblib.dump(xgb, 'models/xgboost_baseline.pkl')
print("Model saved")

Model saved


In [15]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores_xgb = cross_val_score(xgb, X_train_model, y_train, cv=skf, scoring='roc_auc', n_jobs=-1)
print("XGBoost CV AUC scores:", cv_scores_xgb)
print(f"Mean: {cv_scores_xgb.mean():.4f}, Std: {cv_scores_xgb.std():.4f}")

XGBoost CV AUC scores: [0.7532092  0.75442017 0.75761299 0.75609786 0.75626633]
Mean: 0.7555, Std: 0.0015


In [16]:
cv_scores_lgbm = cross_val_score(lgbm, X_train_lgbm, y_train, cv=skf, scoring='roc_auc', n_jobs=-1)
print(f"LightGBM  Mean: {cv_scores_lgbm.mean():.4f}, Std: {cv_scores_lgbm.std():.4f}")

cv_scores_cb = cross_val_score(catboost, X_train_model, y_train, cv=skf, scoring='roc_auc', n_jobs=-1)
print(f"CatBoost  Mean: {cv_scores_cb.mean():.4f}, Std: {cv_scores_cb.std():.4f}")

LightGBM  Mean: 0.7557, Std: 0.0011
CatBoost  Mean: 0.7570, Std: 0.0015


In [17]:
final_results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'XGBoost', 'LightGBM', 'CatBoost'],
    'Single-Split AUC': [0.7410, 0.7186, 0.7615, 0.7606, 0.7611],
    'CV Mean AUC': [None, None, cv_scores_xgb.mean(), cv_scores_lgbm.mean(), cv_scores_cb.mean()],
    'CV Std': [None, None, cv_scores_xgb.std(), cv_scores_lgbm.std(), cv_scores_cb.std()]
})
final_results = final_results.sort_values('CV Mean AUC', ascending=False, na_position='last').reset_index(drop=True)
print(final_results)

final_results.to_csv('reports/model_comparison_results.csv', index=False)
print("Saved successfully")

                 Model  Single-Split AUC  CV Mean AUC    CV Std
0             CatBoost            0.7611     0.756967  0.001512
1             LightGBM            0.7606     0.755678  0.001089
2              XGBoost            0.7615     0.755521  0.001538
3  Logistic Regression            0.7410          NaN       NaN
4        Decision Tree            0.7186          NaN       NaN
Saved successfully
